# Content two-tower transformer: session-based candidate generation

A personalised candidate generator trained on the full `labels_train` pool,
with up to four exposed negatives redrawn per positive each epoch. The
article tower from the content two-tower MLP is unchanged: each article is
encoded from its intrinsic content only (text and entity vectors plus
missing-vector flags). The user tower consumes a sequence of previous sessions
from the same user. Each previous session is represented by the mean-pooled text
and entity vectors of its clicked articles, concatenated into a 164-dimensional
session vector. A transformer encodes the session sequence, masked-mean pools
the valid session tokens and produces the user embedding. The logit is a
scaled normalised dot product of the user and article content embeddings.

Session construction:

- Each impression is one session (`session_id == impression_uid`).
- For a user's consecutive requests, `full_history_rows` is cumulative. The
  suffix appended between request `k` and request `k+1` is exactly the set of
  articles clicked in session `k`; repeated clicks are preserved.
- A request uses only sessions that occurred strictly earlier. The current
  session's clicks are never used.
- Target-split requests may use earlier-split sessions for the same user.

Fallbacks:

- No valid previous session: mean-pool the full causal history (the MLP path).
- No history either: rolling prior-hour popularity order.

Shared protocol:

- Use `full_history_rows` from the request profile for that impression. Never
  reuse a later request's profile for an earlier request.
- Exclude every article in the request's full causal history.
- Eligibility is `available_from <= timestamp`.
- Train on the full `labels_train` rows, tune chronologically on the tail of
  train, then refit on all training rows and evaluate complete validation
  slates.
- Popularity is not a model feature. The rolling prior-hour click order is used
  only for tie-breaking and the empty-history fallback. A request in hour `h`
  only uses the completed hour `h - 1`, matching the rolling-popularity
  baseline.
- The sigmoid is a ranking score. Training uses grouped exposed negatives, so
  it is not a calibrated click probability.

The notebook runs the full pipeline on the complete training and target splits.


## 1. Setup

In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from torch import nn

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
)
CG_DIR = ROOT / "5_Models" / "5_01_Candidate_Generation"
sys.path.insert(0, str(CG_DIR))
sys.path.insert(0, str(ROOT / "src"))

from cf_models import resolve_device
from cg_result_writer import CandidateWriter
from neural_helpers import (
    EarlyStopper,
    EpochNegativeSampler,
    chronological_cutoff,
    chronological_mask,
    max_clicks_per_impression,
    mean_slate_metrics,
    tuning_slates,
)
from newsrec.common import Settings

PROCESSED = ROOT / "1_Data_Preparation" / "processed"
FEATURES = ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
REPRESENTATIONS = ROOT / "3_Representations" / "artifacts"
ARTIFACTS = ROOT / "5_Models" / "5_01_Candidate_Generation" / "artifacts"
RESULTS = ROOT / "results"

TARGET_SPLIT = "validation"
FIT_SPLIT = "train"
LOSS = "bpr"
MODEL_NAME = "content_two_tower_transformer_bpr"
RUN_TAG = "two_tower_transformer_bpr"
ARTIFACT_DIR = ARTIFACTS / RUN_TAG
RECOMMENDATIONS_PATH = ARTIFACT_DIR / f"recommendations_{TARGET_SPLIT}_top500.parquet"
RESULT_PATH = RESULTS / f"candidate_generation_{MODEL_NAME}_{TARGET_SPLIT}.csv"
SESSION_CACHE_DIR = ARTIFACTS / "two_tower_transformer_cache"

M_VALUES = (50, 100, 200, 500, 750, 1000, 1500)
CANDIDATE_CAP = max(M_VALUES)
HIDDEN_DIM = 256
HOUR_NS = 3_600 * 10**9
POPULARITY_WINDOW_HOURS = 1
POPULARITY_SPLITS = (
    ["train", "validation"]
    if TARGET_SPLIT == "validation"
    else ["train", "validation", "test"]
)
EMBED_DIM = 128
DROPOUT = 0.1
SESSION_MAX = 10
TRANSFORMER_LAYERS = 2
TRANSFORMER_HEADS = 4
TRANSFORMER_FF = 512
NUM_EPOCHS = 50
BATCH_SIZE = 16_384
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
SEED = 4222
PATIENCE = 7
TUNE_QUANTILE = 0.9
TUNE_CANDIDATES = PROCESSED / "candidates_train.parquet"
PRIOR_SPLITS = {
    "validation": ["train"],
    "test": ["train", "validation"],
}[TARGET_SPLIT]
TARGET_SOURCE_SPLITS = [*PRIOR_SPLITS, TARGET_SPLIT]
DEVICE = resolve_device("auto")

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
SESSION_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)
print("Target split:", TARGET_SPLIT)
print("Prior splits:", PRIOR_SPLITS)
print("Device:", DEVICE)
print("Torch:", torch.__version__)


### Shared dimensions

In [ ]:
TEXT_DIM = 64
ENTITY_DIM = 100
INPUT_DIM = TEXT_DIM + ENTITY_DIM + 2
SESSION_DIM = TEXT_DIM + ENTITY_DIM


## 2. Data ingestion and model-related processing

### History pooling, session vectors and feature builders

`pool_profiles` mean-pools valid article vectors and L2-normalises each block.
`profile_features` builds the 166-dimensional fallback request input, and
`item_features` builds the 166-dimensional intrinsic article-content input.
There is no popularity input to either tower.

`session_click_suffixes` recovers the clicked article rows of every previous
session from the cumulative `full_history_rows` lists. `load_session_source`
collects the same user's requests across the requested splits, and
`build_session_cache` turns those sessions into 164-dimensional vectors plus a
capped prior-session index.


In [ ]:
def pool_profiles(history_rows, matrix, block=2_000):
    matrix = np.asarray(matrix)
    n = len(history_rows)
    result = np.zeros((n, matrix.shape[1]), dtype=np.float32)
    for start in range(0, n, block):
        chunk = history_rows[start : start + block]
        lengths = np.fromiter(
            (len(rows) for rows in chunk), dtype=np.int64, count=len(chunk)
        )
        if not lengths.sum():
            continue
        flat = np.concatenate(
            [np.asarray(rows, dtype=np.int64) for rows in chunk if len(rows)]
        )
        vectors = np.asarray(matrix[flat], dtype=np.float32)
        valid = np.linalg.norm(vectors, axis=1) > 1e-12
        if not valid.any():
            continue
        groups = np.repeat(np.arange(len(chunk), dtype=np.int64), lengths)
        valid_groups = groups[valid]
        starts = np.r_[
            0, np.flatnonzero(valid_groups[1:] != valid_groups[:-1]) + 1
        ]
        pooled = np.add.reduceat(vectors[valid], starts, axis=0)
        counts = np.diff(np.r_[starts, len(valid_groups)])
        pooled /= counts[:, None]
        norms = np.linalg.norm(pooled, axis=1)
        nonzero = norms > 0
        pooled[nonzero] /= norms[nonzero, None]
        result[start : start + len(chunk)][valid_groups[starts]] = pooled
    return result

In [ ]:
def profile_features(history_rows, text, entity, block=2_000):
    n = len(history_rows)
    out = np.zeros((n, INPUT_DIM), dtype=np.float32)
    for start in range(0, n, block):
        chunk = history_rows[start : start + block]
        text_profile = pool_profiles(chunk, text, block=len(chunk))
        entity_profile = pool_profiles(chunk, entity, block=len(chunk))
        stop = start + len(chunk)
        out[start:stop, :TEXT_DIM] = text_profile
        out[start:stop, TEXT_DIM : TEXT_DIM + ENTITY_DIM] = entity_profile
        out[start:stop, -2] = np.linalg.norm(text_profile, axis=1) > 0
        out[start:stop, -1] = np.linalg.norm(entity_profile, axis=1) > 0
    return out

In [ ]:
def item_features(text, entity):
    text = np.asarray(text, dtype=np.float32)
    entity = np.asarray(entity, dtype=np.float32)
    out = np.zeros((text.shape[0], INPUT_DIM), dtype=np.float32)
    out[:, :TEXT_DIM] = text
    out[:, TEXT_DIM : TEXT_DIM + ENTITY_DIM] = entity
    out[:, -2] = np.linalg.norm(text, axis=1) > 0
    out[:, -1] = np.linalg.norm(entity, axis=1) > 0
    return out


def item_inputs(rows):
    return np.asarray(CONTENT_INPUTS[rows], dtype=np.float32)


### Request profiles

Each impression's profile stores the articles clicked before that request in
`full_history_rows`.

In [ ]:
def load_profiles(split):
    parts = sorted((FEATURES / f"request_profiles_{split}").glob("*.parquet"))
    frame = pl.read_parquet(
        parts,
        columns=["impression_uid", "user_code", "timestamp", "full_history_rows"],
    )
    return frame.sort("timestamp", maintain_order=True)

In [ ]:
TEXT = np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r")
ENTITY = np.load(REPRESENTATIONS / "article_entity100.npy", mmap_mode="r")
TRAIN_PROFILES = load_profiles(FIT_SPLIT)
TRAIN_PROFILE_UIDS = TRAIN_PROFILES["impression_uid"].to_numpy()
PROFILE_INDEX_TRAIN = pl.DataFrame(
    {
        "impression_uid": TRAIN_PROFILE_UIDS,
        "profile_row": np.arange(len(TRAIN_PROFILE_UIDS), dtype=np.int32),
    }
)
TRAIN_PROFILE_INDEX = TRAIN_PROFILES.select("impression_uid", "timestamp")
print(f"Train requests: {TRAIN_PROFILES.height:,}")
del TRAIN_PROFILES

### Causal session-sequence caches

Each request row stores its own session vector, derived from the clicks that
appear in the next request's causal history. Prior-session references are stored
as a CSR-style offset/index pair capped at the `SESSION_MAX` most recent valid
sessions. The same cache stores the 166-dimensional mean-pooled history used by
the fallback path.

The train cache uses train requests only. The target cache combines earlier
splits with the target split for target users, so a validation request can use
that user's train sessions. Session vectors are always strictly earlier than the
request that consumes them.

In [ ]:
def session_click_suffixes(frame):
    rows = frame["full_history_rows"].to_list()
    users = frame["user_code"].to_numpy()
    n = len(rows)
    suffixes = [[] for _ in range(n)]
    starts = np.flatnonzero(np.r_[True, users[1:] != users[:-1]])
    ends = np.r_[starts[1:], n]
    for start, end in zip(starts.tolist(), ends.tolist()):
        for k in range(start + 1, end):
            previous = rows[k - 1]
            current = rows[k]
            if len(current) > len(previous):
                suffixes[k - 1] = current[len(previous) :]
    return suffixes


def build_prior_session_index(users, valid):
    """Build capped CSR-style prior-session references for each request row."""
    users = np.asarray(users)
    valid = np.asarray(valid, dtype=bool)
    n = len(valid)
    starts = np.flatnonzero(np.r_[True, users[1:] != users[:-1]])
    ends = np.r_[starts[1:], n]
    counts = np.zeros(n, dtype=np.int64)
    for start, end in zip(starts.tolist(), ends.tolist()):
        group = np.arange(start, end)
        for i in range(len(group)):
            lower = max(0, i - SESSION_MAX)
            candidates = group[lower:i]
            counts[group[i]] = int(valid[candidates].sum())
    offsets = np.r_[0, np.cumsum(counts)].astype(np.int64)
    indices = np.empty(int(offsets[-1]), dtype=np.int32)
    for start, end in zip(starts.tolist(), ends.tolist()):
        group = np.arange(start, end)
        for i in range(len(group)):
            lower = max(0, i - SESSION_MAX)
            candidates = group[lower:i]
            candidates = candidates[valid[candidates]]
            row = group[i]
            indices[offsets[row] : offsets[row + 1]] = candidates
    return offsets, indices




In [ ]:
def load_session_source(splits, user_filter=None):
    frames = []
    for order, split in enumerate(splits):
        scan = pl.scan_parquet(
            sorted((FEATURES / f"request_profiles_{split}").glob("*.parquet"))
        ).select(
            [
                "impression_uid",
                "user_code",
                "timestamp",
                "full_history_rows",
            ]
        )
        if user_filter is not None:
            scan = scan.filter(pl.col("user_code").is_in(user_filter))
        frames.append(
            scan.collect().with_columns(
                pl.lit(order, dtype=pl.Int8).alias("split_order")
            )
        )
    frame = pl.concat(frames, how="vertical_relaxed")
    return frame.sort(["user_code", "split_order", "timestamp", "impression_uid"])

In [ ]:
class SessionCache:
    """Aligned session vectors, prior-session CSR references and fallback inputs."""

    def __init__(self, vectors, valid, offsets, indices, fallback, row_map):
        self.vectors = vectors
        self.valid = np.asarray(valid, dtype=bool)
        self.offsets = np.asarray(offsets, dtype=np.int64)
        self.indices = np.asarray(indices, dtype=np.int32)
        self.fallback = fallback
        self.row_map = np.asarray(row_map, dtype=np.int64)
        lengths = self.offsets[1:] - self.offsets[:-1]
        self.profile_has_sessions = lengths[self.row_map] > 0

    def gather(self, profile_rows):
        profile_rows = np.asarray(profile_rows, dtype=np.int64)
        rows = self.row_map[profile_rows]
        starts = self.offsets[rows]
        ends = self.offsets[rows + 1]
        lengths = ends - starts
        batch = len(rows)
        sessions = np.zeros((batch, SESSION_MAX, SESSION_DIM), dtype=np.float32)
        mask = np.zeros((batch, SESSION_MAX), dtype=bool)
        total = int(lengths.sum())
        if total:
            row_index = np.repeat(np.arange(batch, dtype=np.int64), lengths)
            position = np.arange(total, dtype=np.int64) - np.repeat(
                np.cumsum(lengths) - lengths, lengths
            )
            absolute = np.repeat(starts, lengths) + position
            session_rows = self.indices[absolute]
            sessions[row_index, position] = np.asarray(self.vectors[session_rows])
            mask[row_index, position] = True
        fallback = np.asarray(self.fallback[rows], dtype=np.float32)
        return sessions, mask, fallback, lengths > 0

In [ ]:
def session_cache_paths(profile_split):
    stem = f"{profile_split}_k{SESSION_MAX}"
    return {
        "vectors": SESSION_CACHE_DIR / f"session_vectors_{stem}.npy",
        "valid": SESSION_CACHE_DIR / f"session_valid_{stem}.npy",
        "offsets": SESSION_CACHE_DIR / f"prior_offsets_{stem}.npy",
        "indices": SESSION_CACHE_DIR / f"prior_indices_{stem}.npy",
        "fallback": SESSION_CACHE_DIR / f"fallback_features_{stem}.npy",
        "row_map": SESSION_CACHE_DIR / f"profile_row_map_{stem}.npy",
        "profile_uids": SESSION_CACHE_DIR / f"profile_uids_{stem}.npy",
    }


def build_session_cache(profile_split, source_splits, profile_uids, user_filter=None):
    paths = session_cache_paths(profile_split)
    frame = load_session_source(source_splits, user_filter=user_filter)
    n = frame.height
    users = frame["user_code"].to_numpy()
    source_uids = frame["impression_uid"].to_numpy()

    vectors = np.lib.format.open_memmap(
        paths["vectors"], mode="w+", dtype=np.float32, shape=(n, SESSION_DIM)
    )
    fallback = np.lib.format.open_memmap(
        paths["fallback"], mode="w+", dtype=np.float32, shape=(n, INPUT_DIM)
    )
    valid = np.zeros(n, dtype=bool)

    suffixes = session_click_suffixes(frame)
    chunk = 20_000
    for start in range(0, n, chunk):
        end = min(start + chunk, n)
        text = pool_profiles(suffixes[start:end], TEXT)
        entity = pool_profiles(suffixes[start:end], ENTITY)
        block = np.hstack([text, entity]).astype(np.float32)
        vectors[start:end] = block
        valid[start:end] = np.linalg.norm(block, axis=1) > 1e-12
        fallback[start:end] = profile_features(
            frame["full_history_rows"].slice(start, end - start).to_list(),
            TEXT,
            ENTITY,
        )
    vectors.flush()
    fallback.flush()
    del suffixes
    del frame

    offsets, indices = build_prior_session_index(users, valid)

    order = pl.DataFrame({"impression_uid": np.asarray(profile_uids)}).join(
        pl.DataFrame(
            {
                "impression_uid": source_uids,
                "session_row": np.arange(n, dtype=np.int64),
            }
        ),
        on="impression_uid",
        how="left",
    )
    row_map = order["session_row"].to_numpy().astype(np.int64)
    np.save(paths["valid"], valid)
    np.save(paths["offsets"], offsets)
    np.save(paths["indices"], indices)
    np.save(paths["row_map"], row_map)
    np.save(paths["profile_uids"], np.asarray(profile_uids, dtype=np.int64))
    print(f"Built session cache for {profile_split}: {n:,} session rows")
    return SessionCache(vectors, valid, offsets, indices, fallback, row_map)


def load_session_cache(profile_split, source_splits, profile_uids, user_filter=None):
    paths = session_cache_paths(profile_split)
    keys = ("vectors", "valid", "offsets", "indices", "fallback", "row_map", "profile_uids")
    if all(paths[key].exists() for key in keys):
        vectors = np.load(paths["vectors"], mmap_mode="r")
        fallback = np.load(paths["fallback"], mmap_mode="r")
        valid = np.load(paths["valid"])
        offsets = np.load(paths["offsets"])
        indices = np.load(paths["indices"])
        row_map = np.load(paths["row_map"])
        print(f"Loaded session cache for {profile_split}: {len(valid):,} session rows")
        return SessionCache(vectors, valid, offsets, indices, fallback, row_map)
    return build_session_cache(
        profile_split, source_splits, profile_uids, user_filter=user_filter
    )

In [ ]:
TRAIN_SESSION_CACHE = load_session_cache(
    profile_split=FIT_SPLIT,
    source_splits=[FIT_SPLIT],
    profile_uids=TRAIN_PROFILE_UIDS,
)

print(
    "Train requests with a valid prior session: "
    f"{TRAIN_SESSION_CACHE.profile_has_sessions.mean():.3f}"
)

### Article content inputs and the time-eligible catalogue

Article vectors are indexed by `article_row`, with row 0 reserved as padding.
The item tower receives only intrinsic content vectors and missing-vector flags.
Rolling popularity is not an item-tower feature; it is used only for candidate
tie-breaking and the empty-history fallback. The catalogue is ordered by
`available_from` so each request timestamp selects a prefix.


In [ ]:
article_index = pl.read_parquet(
    REPRESENTATIONS / "article_index.parquet", columns=["article_row", "news_code"]
).sort("article_row")
n_items = article_index.height + 1
CONTENT_INPUTS = item_features(TEXT, ENTITY)
availability = pl.read_parquet(
    REPRESENTATIONS / "article_availability.parquet",
    columns=["news_code", "available_from"],
)
catalogue = article_index.join(availability, on="news_code", how="left")
ordered = catalogue.sort(["available_from", "news_code"])
ARTICLE_ROWS_SORTED = ordered["article_row"].to_numpy().astype(np.int64)
NEWS_SORTED = ordered["news_code"].to_numpy().astype(np.int64)
AVAILABLE_SORTED = ordered["available_from"].to_numpy().astype("datetime64[ns]")
POS_IN_SORTED = np.full(n_items, -1, dtype=np.int64)
POS_IN_SORTED[ARTICLE_ROWS_SORTED] = np.arange(
    len(ARTICLE_ROWS_SORTED), dtype=np.int64
)
index_codes = article_index["news_code"].to_numpy().astype(np.int64)
index_rows = article_index["article_row"].to_numpy().astype(np.int64)
row_of_code = np.zeros(int(index_codes.max()) + 1, dtype=np.int64)
row_of_code[index_codes] = index_rows

hourly = (
    pl.concat(
        [
            pl.scan_parquet(PROCESSED / f"candidates_{split}.parquet")
            .select(
                "news_code",
                "clicked",
                (pl.col("timestamp").dt.epoch("ns") // HOUR_NS)
                .cast(pl.Int64)
                .alias("hour"),
            )
            for split in POPULARITY_SPLITS
        ]
    )
    .group_by("news_code", "hour")
    .agg(clk=pl.col("clicked").cast(pl.Int64).sum())
    .collect()
)

source_codes = hourly["news_code"].to_numpy().astype(np.int64)
source_rows = row_of_code[source_codes]

MIN_HOUR = int(hourly["hour"].min())
MAX_HOUR = int(hourly["hour"].max())
N_HOURS = MAX_HOUR - MIN_HOUR + 1
CLICKS = np.zeros((N_HOURS, n_items), dtype=np.int32)
hour_positions = hourly["hour"].to_numpy().astype(np.int64) - MIN_HOUR
CLICKS[hour_positions, source_rows] = hourly["clk"].to_numpy().astype(np.int32)


def hour_index_of(timestamps_ns):
    return np.asarray(timestamps_ns, dtype=np.int64) // HOUR_NS - MIN_HOUR


def hourly_popularity(hour_index, item_rows):
    hour_index = np.asarray(hour_index, dtype=np.int64)
    item_rows = np.asarray(item_rows, dtype=np.int64)
    if hour_index.ndim == 0:
        hour_index = np.full(item_rows.shape, int(hour_index), dtype=np.int64)
    else:
        hour_index, item_rows = np.broadcast_arrays(hour_index, item_rows)
    clk = np.zeros(item_rows.shape, dtype=np.float32)
    for offset in range(POPULARITY_WINDOW_HOURS):
        source = hour_index - 1 - offset
        valid = source >= 0
        if valid.any():
            clk[valid] += CLICKS[source[valid], item_rows[valid]]
    return np.log1p(clk)


print(f"Catalogue articles: {article_index.height:,}")
print(f"Item input shape: {CONTENT_INPUTS.shape}")
print(f"Rolling popularity hours: {N_HOURS:,}")
print(f"Articles clicked at least once: {int((CLICKS.sum(0) > 0).sum()):,}")


### Training groups and chronological tuning slice

Training rows come from the full `labels_train` pool: each clicked positive is
paired with up to four exposed non-clicks redrawn from the same request each
epoch. The last 10% of training timestamps are held out for early stopping; the tuning slates come
from the complete `candidates_train.parquet`.

In [ ]:
run_manifest = json.loads((FEATURES / "run_manifest.json").read_text())
if run_manifest.get("training_candidate_pool") != "complete":
    raise ValueError(
        "Per-epoch sampling needs the complete training pool. "
        "Rebuild stage 4 with the default training_negative_sampling='epoch'."
    )
CFG = Settings(**run_manifest["config"])

TRAINING_ROWS = (
    pl.read_parquet(sorted((FEATURES / "labels_train").glob("*.parquet")))
    .select("impression_uid", "news_code", "clicked")
    .to_pandas()
)
if TRAINING_ROWS["clicked"].isna().any() or not TRAINING_ROWS[
    "clicked"
].isin([0, 1]).all():
    raise ValueError("Training rows must be uniquely binary-labelled")

profile_lookup = PROFILE_INDEX_TRAIN.join(
    TRAIN_PROFILE_INDEX, on="impression_uid", how="left"
)

positive_uids = np.unique(
    TRAINING_ROWS.loc[TRAINING_ROWS["clicked"].eq(1), "impression_uid"].to_numpy()
)
positive_profiles = profile_lookup.filter(
    pl.col("impression_uid").is_in(positive_uids)
)
cutoff_ns = chronological_cutoff(
    positive_profiles["timestamp"].to_numpy(), TUNE_QUANTILE
)
fit_profiles = chronological_mask(profile_lookup["timestamp"].to_numpy(), cutoff_ns)
fit_uids = set(
    profile_lookup.filter(pl.Series("fit", fit_profiles))["impression_uid"].to_list()
)
FIT_ROWS = TRAINING_ROWS[
    TRAINING_ROWS["impression_uid"].isin(fit_uids)
].reset_index(drop=True)
ALL_ROWS = TRAINING_ROWS
n_groups = int(TRAINING_ROWS["clicked"].sum())
EPOCH_SAMPLER = EpochNegativeSampler(ALL_ROWS, CFG)


def sample_epoch(epoch, *, fit_only):
    """Draw exposed negatives for one epoch and build transformer arrays."""
    groups, _ = EPOCH_SAMPLER.draw(
        int(epoch), uids=fit_uids if fit_only else None
    )
    groups = (
        pl.from_pandas(groups)
        .join(profile_lookup, on="impression_uid", how="left")
        .with_row_index("group_row")
        .join(
            article_index.select(
                pl.col("news_code"),
                pl.col("article_row").alias("pos_row"),
            ),
            left_on="positive_news_code",
            right_on="news_code",
            how="left",
        )
    )
    if groups["profile_row"].null_count() or groups["pos_row"].null_count():
        raise ValueError("Sampled groups are missing training profiles or article rows")
    negative_long = (
        groups.select("group_row", "negative_news_codes")
        .explode("negative_news_codes")
        .rename({"negative_news_codes": "negative_news_code"})
        .drop_nulls("negative_news_code")
        .with_columns(pl.int_range(pl.len()).over("group_row").alias("neg_pos"))
        .join(
            article_index.select(
                pl.col("news_code").alias("negative_news_code"),
                pl.col("article_row").alias("neg_row"),
            ),
            on="negative_news_code",
            how="left",
        )
        .sort(["group_row", "neg_pos"])
    )
    if negative_long["neg_row"].null_count():
        raise ValueError("Sampled negatives are missing article rows")
    n_epoch_groups = groups.height
    NEG_MAX = int(groups["negative_news_codes"].list.len().max() or 0)
    if n_epoch_groups and NEG_MAX == 0:
        raise ValueError("Sampled training rows contain no exposed negatives")
    NEG_MATRIX = np.zeros((n_epoch_groups, NEG_MAX), dtype=np.int64)
    group_rows = negative_long["group_row"].to_numpy().astype(np.int64)
    NEG_MATRIX[
        group_rows, negative_long["neg_pos"].to_numpy().astype(np.int64)
    ] = negative_long["neg_row"].to_numpy().astype(np.int64)
    NEG_LEN = np.bincount(group_rows, minlength=n_epoch_groups).astype(np.int64)
    return (
        groups["profile_row"].to_numpy().astype(np.int32),
        groups["pos_row"].to_numpy().astype(np.int32),
        NEG_MATRIX,
        NEG_LEN,
    )


TUNE_NDCG_K = max_clicks_per_impression(TUNE_CANDIDATES)

tune = tuning_slates(
    TUNE_CANDIDATES,
    cutoff_ns,
    article_index,
    PROFILE_INDEX_TRAIN,
    max_impressions=None,
    seed=SEED + 1,
)
TUNE_PROFILE_ROWS = tune["profile_row"].to_numpy().astype(np.int32)
TUNE_ITEM_ROWS = tune["article_row"].to_numpy().astype(np.int32)
TUNE_CLICKED = tune["clicked"].to_numpy().astype(np.float32)
TUNE_UIDS = tune["impression_uid"].to_numpy()
print(f"Training rows: {len(TRAINING_ROWS):,}")
print(f"Training positives: {n_groups:,}")
print(f"Chronological fit rows: {len(FIT_ROWS):,} / {len(ALL_ROWS):,}")
print(f"Sampler negatives per positive: up to {CFG.negative_count}")
print("Tuning cutoff:", np.datetime64(cutoff_ns, "ns"))
print(f"Tuning impressions: {tune['impression_uid'].n_unique():,}")
print(f"Tuning pairs: {tune.height:,}")
print(f"Tuning NDCG k (max training clicks per impression): {TUNE_NDCG_K}")


## 3. Setting up the model

### Session two-tower network definition

`SessionTwoTower` keeps the MLP item tower unchanged. The user side has three
parts: a projection from 164-dimensional session vectors to the transformer
width, learned positional embeddings, and a transformer encoder. Valid session
tokens are masked-mean pooled and projected to the 128-dimensional user
embedding. A separate 166-dimensional fallback tower encodes mean-pooled history
when a request has no valid prior session. Both user paths are L2-normalised.
The item tower consumes the 166-dimensional intrinsic content input. The
outputs are combined by a scaled dot product. There are no user-ID, article-ID,
popularity or learned per-article bias terms.


In [ ]:
class SessionTwoTower(nn.Module):
    def __init__(
        self,
        item_input_dim=INPUT_DIM,
        session_input_dim=SESSION_DIM,
        fallback_input_dim=INPUT_DIM,
        hidden_dim=256,
        embed_dim=128,
        dropout=0.1,
        scale=10.0,
        max_scale=100.0,
        session_max=SESSION_MAX,
        transformer_layers=TRANSFORMER_LAYERS,
        transformer_heads=TRANSFORMER_HEADS,
        transformer_ff=TRANSFORMER_FF,
    ):
        super().__init__()
        self.session_max = int(session_max)
        self.session_proj = nn.Sequential(
            nn.Linear(session_input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.position = nn.Embedding(self.session_max, hidden_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=transformer_heads,
            dim_feedforward=transformer_ff,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.session_encoder = nn.TransformerEncoder(
            layer, num_layers=transformer_layers, enable_nested_tensor=False
        )
        self.session_head = nn.Sequential(
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
        )
        self.fallback_tower = self._tower(
            fallback_input_dim, hidden_dim, embed_dim, dropout
        )
        self.item_tower = self._tower(
            item_input_dim, hidden_dim, embed_dim, dropout
        )
        self.logit_scale = nn.Parameter(
            torch.tensor(float(np.log(scale)), dtype=torch.float32)
        )
        self.max_log_scale = float(np.log(max_scale))

    @staticmethod
    def _tower(input_dim, hidden_dim, embed_dim, dropout):
        return nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
        )

    def encode_sessions(self, sessions, session_mask):
        mask = session_mask.to(torch.bool)
        safe_mask = mask.clone()
        empty = ~safe_mask.any(dim=1)
        if empty.any():
            safe_mask[empty, 0] = True
            sessions = sessions.clone()
            sessions[empty, 0] = 0.0
        positions = torch.arange(sessions.size(1), device=sessions.device)
        hidden = self.session_proj(sessions)
        hidden = hidden + self.position(positions).unsqueeze(0)
        hidden = self.session_encoder(hidden, src_key_padding_mask=~safe_mask)
        weights = safe_mask.unsqueeze(-1).to(hidden.dtype)
        pooled = (hidden * weights).sum(dim=1) / weights.sum(dim=1).clamp_min(1.0)
        return F.normalize(self.session_head(pooled), dim=-1)

    def encode_users(self, sessions, session_mask, fallback, has_sessions):
        session_users = self.encode_sessions(sessions, session_mask)
        fallback_users = F.normalize(self.fallback_tower(fallback), dim=-1)
        return torch.where(
            has_sessions.unsqueeze(-1).to(torch.bool),
            session_users,
            fallback_users,
        )

    def encode_items(self, values):
        return F.normalize(self.item_tower(values), dim=-1)

    def forward(
        self,
        sessions,
        session_mask,
        fallback,
        has_sessions,
        item_values,
    ):
        scale = torch.exp(torch.clamp(self.logit_scale, max=self.max_log_scale))
        return scale * (
            self.encode_users(sessions, session_mask, fallback, has_sessions)
            * self.encode_items(item_values)
        ).sum(dim=-1)

In [ ]:
def user_tensors(cache, profile_rows):
    sessions, session_mask, fallback, has_sessions = cache.gather(profile_rows)
    return (
        torch.as_tensor(sessions, device=DEVICE),
        torch.as_tensor(session_mask, device=DEVICE),
        torch.as_tensor(fallback, device=DEVICE),
        torch.as_tensor(has_sessions, device=DEVICE),
    )

## 4. Training the model

### Session two-tower training, chronological tuning and refit

The user tower consumes the previous-session sequence, with the mean-pooled
history fallback. The item tower consumes the 166-dimensional intrinsic content
input. The towers do not share weights; their outputs are L2-normalised and
combined by a scaled dot product. Early stopping uses mean slate NDCG@k on the
chronological tuning slice,
with k set to the maximum number of clicks per impression in the training
slates; the final model is then refit on all training rows for the selected
epoch count. BPR scores the positive against the sampled negatives each epoch.


In [ ]:
def tune_ndcg(model):
    model.eval()
    scores = np.empty(len(TUNE_CLICKED), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(TUNE_CLICKED), 20_000):
            end = min(start + 20_000, len(TUNE_CLICKED))
            users = user_tensors(
                TRAIN_SESSION_CACHE, TUNE_PROFILE_ROWS[start:end]
            )
            items = torch.as_tensor(
                item_inputs(TUNE_ITEM_ROWS[start:end]),
                device=DEVICE,
            )
            scores[start:end] = model(*users, items).cpu().numpy()
    _, ndcg = mean_slate_metrics(
        TUNE_UIDS, TUNE_CLICKED, scores, TUNE_NDCG_K
    )
    return ndcg


In [ ]:
def train_epoch(
    model,
    optimizer,
    profile_rows,
    pos_item_rows,
    neg_matrix,
    neg_len,
    rng,
):
    model.train()
    order = rng.permutation(len(profile_rows))
    total = 0.0
    for start in range(0, len(order), BATCH_SIZE):
        batch = order[start : start + BATCH_SIZE]
        users = user_tensors(TRAIN_SESSION_CACHE, profile_rows[batch])
        positives = torch.as_tensor(
            item_inputs(pos_item_rows[batch]),
            device=DEVICE,
        )
        neg_rows_np = neg_matrix[batch]
        neg_lens_np = neg_len[batch]
        negatives = torch.as_tensor(
            neg_rows_np, dtype=torch.long, device=DEVICE
        )
        neg_lens = torch.as_tensor(
            neg_lens_np, dtype=torch.long, device=DEVICE
        )
        neg_mask = (
            torch.arange(negatives.shape[1], device=DEVICE)[None, :]
            < neg_lens[:, None]
        )
        negative_inputs = torch.as_tensor(
            item_inputs(neg_rows_np.reshape(-1)),
            device=DEVICE,
        )
        optimizer.zero_grad(set_to_none=True)
        user_repr = model.encode_users(*users)
        scale = torch.exp(
            torch.clamp(model.logit_scale, max=model.max_log_scale)
        )
        positive_scores = scale * (
            user_repr * model.encode_items(positives)
        ).sum(dim=-1)
        negative_item_repr = model.encode_items(negative_inputs).reshape(
            len(batch), negatives.shape[1], -1
        )
        negative_scores = scale * (
            user_repr[:, None, :] * negative_item_repr
        ).sum(dim=-1)
        pair_loss = F.logsigmoid(positive_scores[:, None] - negative_scores)
        loss = -(pair_loss * neg_mask).sum() / neg_mask.sum()
        loss.backward()
        optimizer.step()
        total += float(loss.item()) * len(batch)
    return total / len(order)


In [ ]:
def build_model():
    return SessionTwoTower(
        item_input_dim=INPUT_DIM,
        session_input_dim=SESSION_DIM,
        fallback_input_dim=INPUT_DIM,
        hidden_dim=HIDDEN_DIM,
        embed_dim=EMBED_DIM,
        dropout=DROPOUT,
        session_max=SESSION_MAX,
        transformer_layers=TRANSFORMER_LAYERS,
        transformer_heads=TRANSFORMER_HEADS,
        transformer_ff=TRANSFORMER_FF,
    ).to(DEVICE)

In [ ]:
model = build_model()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
rng = np.random.default_rng(SEED + 1)
training_log = []
stopper = EarlyStopper(patience=PATIENCE)
for epoch in range(1, NUM_EPOCHS + 1):
    started = time.time()
    profile_rows, pos_rows, neg_matrix, neg_len = sample_epoch(
        epoch - 1, fit_only=True
    )
    loss = train_epoch(
        model,
        optimizer,
        profile_rows,
        pos_rows,
        neg_matrix,
        neg_len,
        rng,
    )
    ndcg = tune_ndcg(model)
    elapsed = time.time() - started
    training_log.append(
        {
            "phase": "tune",
            "epoch": epoch,
            "train_loss": loss,
            "tune_ndcg": ndcg,
            "ndcg_k": TUNE_NDCG_K,
            "seconds": elapsed,
        }
    )
    print(
        f"epoch {epoch:02d} loss={loss:.6f} "
        f"tune_ndcg@{TUNE_NDCG_K}={ndcg:.5f} ({elapsed:.1f}s)"
    )
    if stopper.step(ndcg, model, epoch):
        print("Early stopping")
        break

stopper.restore(model)
print(
    f"Best epoch: {stopper.best_epoch} "
    f"(tune_ndcg@{TUNE_NDCG_K}={stopper.best_score:.5f})"
)

final_model = build_model()
optimizer = torch.optim.AdamW(
    final_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
rng = np.random.default_rng(SEED + 2)
for epoch in range(1, stopper.best_epoch + 1):
    started = time.time()
    profile_rows, pos_rows, neg_matrix, neg_len = sample_epoch(
        epoch - 1, fit_only=False
    )
    loss = train_epoch(
        final_model,
        optimizer,
        profile_rows,
        pos_rows,
        neg_matrix,
        neg_len,
        rng,
    )
    elapsed = time.time() - started
    training_log.append(
        {
            "phase": "refit",
            "epoch": epoch,
            "train_loss": loss,
            "tune_ndcg": float("nan"),
            "ndcg_k": TUNE_NDCG_K,
            "seconds": elapsed,
        }
    )
    print(f"refit epoch {epoch:02d} loss={loss:.6f} ({elapsed:.1f}s)")

model = final_model
torch.save(
    {
        "state_dict": model.state_dict(),
        "config": {
            "user_tower": "session_transformer_with_history_fallback",
            "session_input_dim": SESSION_DIM,
            "session_max": SESSION_MAX,
            "transformer_layers": TRANSFORMER_LAYERS,
            "transformer_heads": TRANSFORMER_HEADS,
            "transformer_feedforward": TRANSFORMER_FF,
            "session_pooling": "masked_mean",
            "fallback_input_dim": INPUT_DIM,
            "item_input_dim": INPUT_DIM,
            "hidden_dim": HIDDEN_DIM,
            "embed_dim": EMBED_DIM,
            "dropout": DROPOUT,
            "popularity_usage": "tie-break and cold or short-list fallback only",
            "popularity_source": "rolling prior-hour click order",
            "popularity_window_hours": POPULARITY_WINDOW_HOURS,
            "popularity_hour_size_ns": HOUR_NS,
            "popularity_splits": POPULARITY_SPLITS,
            "n_hours": N_HOURS,
            "objective": LOSS,
            "max_epochs": NUM_EPOCHS,
            "best_epoch": stopper.best_epoch,
            "epochs": stopper.best_epoch,
            "tune_ndcg": stopper.best_score,
            "ndcg_k": TUNE_NDCG_K,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "seed": SEED,
            "training_data": "labels_train full training pool",
            "train_groups": int(n_groups),
            "negative_sampling": "per-epoch exposed negatives",
            "negative_count": int(CFG.negative_count),
            "sampler_epoch_base": 0,
            "fit_rows": int(len(FIT_ROWS)),
            "refit_rows": int(len(ALL_ROWS)),
        },
    },
    ARTIFACT_DIR / "model.pt",
)
pl.DataFrame(training_log).write_csv(ARTIFACT_DIR / "training_log.csv")
print("Saved model:", ARTIFACT_DIR / "model.pt")


## 5. Results

### Candidate helpers

These helpers keep the candidate format, eligibility policy, history exclusion
and popularity tie-breaking consistent with the other candidate generators.

In [ ]:
def popularity_select(order, n_eligible, excluded=None, k=None):
    if k is None:
        k = CANDIDATE_CAP
    excluded_set = set() if excluded is None else set(map(int, excluded))
    selected = []
    for position in order:
        position = int(position)
        if position >= n_eligible or position in excluded_set:
            continue
        selected.append(position)
        if len(selected) == k:
            return np.asarray(selected, dtype=np.int64)
    raise ValueError(
        f"only {len(selected)} popular eligible articles; cannot select {k}"
    )

In [ ]:
def popularity_fill(available, popularity, news_code, count):
    pool = np.flatnonzero(available)
    if pool.size < count:
        raise ValueError(
            f"only {pool.size} eligible articles remain; cannot fill {count}"
        )
    order = np.lexsort((news_code[pool], -popularity[pool]))
    return pool[order[:count]]

In [ ]:
def select_top_k(
    scores, popularity, news_code, k, excluded=None, fallback_order=None
):
    scores = np.asarray(scores, dtype=np.float32)
    popularity = np.asarray(popularity)
    news_code = np.asarray(news_code)
    if scores.ndim != 1:
        raise ValueError("scores must be one-dimensional")
    if popularity.shape != scores.shape or news_code.shape != scores.shape:
        raise ValueError("score, popularity and news_code shapes must match")
    if k < 1:
        raise ValueError("k must be positive")
    n = scores.size
    if n < k:
        raise ValueError(f"only {n} eligible articles; cannot select {k}")
    if fallback_order is not None:
        return popularity_select(fallback_order, n, excluded=excluded, k=k)

    available = np.ones(n, dtype=bool)
    if excluded is not None and len(excluded):
        positions = np.asarray(excluded, dtype=np.int64)
        if positions.min() < 0 or positions.max() >= n:
            raise ValueError("excluded position outside the eligible array")
        available[positions] = False
        scores = scores.copy()
        scores[positions] = -np.inf

    finite = np.isfinite(scores)
    n_finite = int(np.count_nonzero(finite))
    if n_finite == 0:
        return popularity_fill(available, popularity, news_code, k)

    take = min(k, n_finite)
    if take < n_finite:
        frontier = np.argpartition(-scores, take - 1)[:take]
        threshold = scores[frontier].min()
    else:
        threshold = scores[finite].min()

    above = np.flatnonzero(scores > threshold)
    need = take - above.size
    ties = np.flatnonzero(scores == threshold)
    if need > 0:
        if ties.size > need:
            tie_order = np.lexsort((news_code[ties], -popularity[ties]))
            ties = ties[tie_order[:need]]
        chosen_positions = np.r_[above, ties]
    else:
        chosen_positions = above

    order = np.lexsort(
        (
            news_code[chosen_positions],
            -popularity[chosen_positions],
            -scores[chosen_positions],
        )
    )
    selected = chosen_positions[order].astype(np.int64)

    if selected.size < k:
        remaining = available.copy()
        remaining[selected] = False
        fill = popularity_fill(remaining, popularity, news_code, k - selected.size)
        selected = np.r_[selected, fill]

    return selected

In [ ]:
def encode_user_batches(model, cache, profile_rows, batch=20_000):
    profile_rows = np.asarray(profile_rows, dtype=np.int64)
    out = np.empty((len(profile_rows), EMBED_DIM), dtype=np.float32)
    model.eval()
    with torch.no_grad():
        for start in range(0, len(profile_rows), batch):
            end = min(start + batch, len(profile_rows))
            users = user_tensors(cache, profile_rows[start:end])
            out[start:end] = model.encode_users(*users).cpu().numpy()
    return out

In [ ]:
def encode_item_batches(model, features, batch=100_000):
    out = np.empty((len(features), EMBED_DIM), dtype=np.float32)
    model.eval()
    with torch.no_grad():
        for start in range(0, len(features), batch):
            end = min(start + batch, len(features))
            values = torch.from_numpy(
                np.array(features[start:end], dtype=np.float32, copy=True)
            ).to(DEVICE)
            out[start:end] = model.encode_items(values).cpu().numpy()
    return out

### Score the catalogue and write Top-1500 candidates

The item tower is evaluated once for the availability-sorted catalogue. Each
timestamp batch builds user embeddings from the target session cache and
computes a user-by-candidate score matrix. Articles in the request's full
causal history
are excluded; requests with neither a valid prior session nor history use the
rolling popularity order. Ties are broken by popularity, then `news_code`.

In [ ]:
print(f"[{LOSS}] Scoring {MODEL_NAME}")
target_profiles = load_profiles(TARGET_SPLIT)
TARGET_PROFILE_UIDS = target_profiles["impression_uid"].to_numpy()
TARGET_SESSION_CACHE = load_session_cache(
    profile_split=TARGET_SPLIT,
    source_splits=TARGET_SOURCE_SPLITS,
    profile_uids=TARGET_PROFILE_UIDS,
    user_filter=target_profiles["user_code"].unique().to_list(),
)
model.eval()
user_emb = encode_user_batches(
    model, TARGET_SESSION_CACHE, np.arange(len(TARGET_PROFILE_UIDS))
)
target_fallback = np.asarray(
    TARGET_SESSION_CACHE.fallback[TARGET_SESSION_CACHE.row_map], dtype=np.float32
)
has_profile = (target_fallback[:, -2] > 0) | (target_fallback[:, -1] > 0)
print(
    "Target requests with prior sessions: "
    f"{TARGET_SESSION_CACHE.profile_has_sessions.mean():.3f}"
)
print(f"Target requests with any history: {has_profile.mean():.3f}")

impression_uids = target_profiles["impression_uid"].to_numpy()
timestamps = target_profiles["timestamp"].to_numpy().astype("datetime64[ns]")
profile_hours = hour_index_of(timestamps.astype(np.int64))
history_rows = target_profiles["full_history_rows"].to_list()
starts = np.flatnonzero(np.r_[True, timestamps[1:] != timestamps[:-1]])
ends = np.r_[starts[1:], len(timestamps)]
batches = list(zip(starts, ends))
processed_requests = int(sum(end - start for start, end in batches))
print(f"Scoring requests: {processed_requests:,} / {len(timestamps):,}")

cutoff_values = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)
writer = CandidateWriter(RECOMMENDATIONS_PATH)
current_hour = None
item_emb_cached = encode_item_batches(
    model, item_inputs(ARTICLE_ROWS_SORTED)
)
popularity_sorted = None
fallback_order = None
started = time.time()
try:
    for batch_no, (start, end) in enumerate(batches, 1):
        timestamp = timestamps[start]
        batch_hour = int(profile_hours[start])
        if batch_hour != current_hour:
            popularity_sorted = hourly_popularity(
                batch_hour, ARTICLE_ROWS_SORTED
            )
            fallback_order = np.lexsort((NEWS_SORTED, -popularity_sorted))
            current_hour = batch_hour
        n_eligible = int(
            np.searchsorted(AVAILABLE_SORTED, timestamp, side="right")
        )
        scores = user_emb[start:end] @ item_emb_cached[:n_eligible].T
        popularity_batch = popularity_sorted[:n_eligible]
        news_batch = NEWS_SORTED[:n_eligible]
        default_fallback = fallback_order
        selected_batch = np.empty(
            (end - start, CANDIDATE_CAP), dtype=np.int64
        )
        for local, profile_row in enumerate(range(start, end)):
            rows = history_rows[profile_row]
            if rows:
                excluded = POS_IN_SORTED[np.asarray(rows, dtype=np.int64)]
                excluded = excluded[
                    (excluded >= 0) & (excluded < n_eligible)
                ]
            else:
                excluded = np.empty(0, dtype=np.int64)
            fallback = (
                None if has_profile[profile_row] else default_fallback
            )
            selected_batch[local] = select_top_k(
                scores[local],
                popularity_batch,
                news_batch,
                CANDIDATE_CAP,
                excluded=excluded,
                fallback_order=fallback,
            )

        uids = impression_uids[start:end]
        writer.add(
            np.repeat(uids, CANDIDATE_CAP),
            np.tile(cutoff_values, end - start),
            news_batch[selected_batch.reshape(-1)],
        )
        if batch_no % 1000 == 0 or batch_no == len(batches):
            print(
                f"Scored {batch_no:,}/{len(batches):,} timestamp batches; "
                f"{writer.rows:,} rows",
                flush=True,
            )
finally:
    writer.close()

elapsed = time.time() - started
actual_rows = (
    pl.scan_parquet(RECOMMENDATIONS_PATH)
    .select(pl.len())
    .collect()
    .item()
)
print(f"Wrote {actual_rows:,} candidate rows in {elapsed:.1f}s")
print("Candidates:", RECOMMENDATIONS_PATH)


### Evaluation

Uses the shared `CandidateEvaluator`, which validates impression coverage,
cutoffs, uniqueness and time eligibility.

In [ ]:
from cg_evaluation import CandidateEvaluator

evaluator = CandidateEvaluator(root=ROOT, split=TARGET_SPLIT)
evaluation = evaluator.evaluate(
    RECOMMENDATIONS_PATH,
    model_name=MODEL_NAME,
    m_values=M_VALUES,
)
evaluation.write_csv(RESULT_PATH)
print(f"[{LOSS}]")
print(evaluation.sort("m"))
print("Results:", RESULT_PATH)


## Notes and limitations

- The user representation is specific to each request. Repeat users do not
  share a single latest profile; earlier requests use only earlier feedback.
- A session is one impression. Its vector is the mean-pooled text and entity
  content of the articles clicked in that session, recovered from the suffix
  appended to the next request's `full_history_rows`.
- Prior sessions may come from earlier splits for the same user, but never from
  the current or a future request. The current session's clicks are not used.
- The transformer path is used only when at least one prior session has a
  non-zero vector. Otherwise the model falls back to the MLP mean-pooled
  history tower, then to the rolling popularity order when history is empty.
- This is a pure content model: the item tower sees only the static text and
  entity vectors plus missing-vector flags. Availability is used solely for
  eligibility. There are no user-ID, article-ID, popularity or per-article
  bias inputs.
- Popularity is used only for tie-breaking and empty-history or short-list
  fallback. A request in hour `h` uses only completed hours before `h`, so no
  target-period or same-hour information enters the score.
- Excluding the causal history removes repeat-click opportunities, matching the
  content baseline. Recall therefore does not measure repeat clicks.
- Training uses a chronological tuning slice, early stopping and a final refit
  on all training rows.
- The sigmoid is a ranking score. Grouped exposed negatives make the training
  positive rate much higher than held-out CTR, so the output is not calibrated.
- `available_from` is the project's observable-availability proxy, not a
  verified publication timestamp.
